# CV parser: speed test of the three small models
Measures how fast each fine-tuned model turns a CV page image into JSON, and how much memory it needs.
- **Runtime:** start on **CPU** (free; 2 cores, ~12 GB RAM — slower than most laptops, so these numbers are a worst case).
  Optionally run again on an **L4** GPU (set `DEVICE = "cuda"` in the config cell). Not T4: the scorer uses bf16 on GPU, which T4 lacks.
- **Inputs** (on Drive, `MyDrive/cv_parser_colab/speed/`): `models/` (InternVL3-1B average of 10 runs, SmolVLM-500M epoch 9,
  granite-docling-258M epoch 5) and `bundle/` (the project's `src/evaluate.py` + 10 validation pages with labels).
- **Output:** a results table and chart here, and `results_<device>.csv` in the same Drive folder.

## 1. Settings
Everything you might change is in this cell. `N_PAGES` pages per model; on CPU expect roughly 1–5 min per page for InternVL,
less for the two smaller models, so 10 pages × 3 models takes about 1–2 h. Use `N_PAGES = 3` for a quick look.

In [ ]:
DEVICE = "cpu"        # "cpu" or "cuda"
N_PAGES = 10          # validation pages per model (max 10 in the bundle)
DRIVE = "/content/drive/MyDrive/cv_parser_colab/speed"
WORK = "/content/speed"
# name: (folder under models/, scoring arguments used on TRUBA for that model)
MODELS = {
    "InternVL3-1B (average of 10)": ("internvl_soup10", "--max-patches 12"),
    "SmolVLM-500M (ep9)":           ("smolvlm_ep9",     "--max-patches 12 --max-new-tokens 3000"),
    "granite-docling-258M (ep5)":   ("granite_ep5",     "--max-patches 12 --max-new-tokens 3000"),
}
print(f"{DEVICE}, {N_PAGES} pages x {len(MODELS)} models")

## 2. Set up (about 3–5 min)
Mounts Drive, copies the code, pages and models to the local disk (reading weights from Drive is slow), and installs the
library versions the models were trained with (transformers 4.57.6, peft 0.21.2). Good result: the last line lists the
CPU count, RAM, GPU (or none) and the three model folders. If a model folder is missing, check the Drive upload.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install transformers==4.57.6 peft==0.21.2
!mkdir -p {WORK} && cp -r {DRIVE}/bundle/. {WORK}/ && cp -r {DRIVE}/models {WORK}/
import os, torch, transformers, psutil
gpu = torch.cuda.get_device_name() if torch.cuda.is_available() else "none"
print(f"transformers {transformers.__version__}, torch {torch.__version__}, {os.cpu_count()} CPUs, "
      f"{psutil.virtual_memory().total / 2**30:.0f} GB RAM, GPU {gpu}, models: {sorted(os.listdir(WORK + '/models'))}")
assert DEVICE == "cpu" or torch.cuda.is_available(), "DEVICE is cuda but this runtime has no GPU"

## 3. Run each model (the long step)
Runs the project's own scorer, `src/evaluate.py`, once per model on the same pages, so the prompt and decoding are exactly
those used for the reported scores. `/usr/bin/time -v` records the process's peak memory. Each page prints a line with its
status, field F1, output tokens and seconds. If a model fails, its error is printed and the loop goes on to the next model.

In [ ]:
import subprocess, time, re, json
results = {}
for name, (folder, extra) in MODELS.items():
    out = f"{WORK}/results/{folder}_{DEVICE}"
    cmd = (f"cd {WORK} && /usr/bin/time -v python src/evaluate.py --model models/{folder} --set val --limit {N_PAGES} "
           f"--device {DEVICE} --batch 1 --out {out} {extra}")
    print(f"== {name}", flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    wall = time.time() - t0
    print("\n".join(l for l in p.stdout.splitlines() if l.startswith("  ")))
    if p.returncode != 0:
        print(f"FAILED (exit {p.returncode}):\n{p.stderr[-2000:]}"); continue
    rss = int(re.search(r"Maximum resident set size \(kbytes\): (\d+)", p.stderr).group(1)) / 2**20
    results[name] = {"folder": folder, "wall_s": wall, "peak_ram_gb": rss}
    print(f"   done in {wall / 60:.1f} min, peak RAM {rss:.1f} GB", flush=True)
print(f"{len(results)} of {len(MODELS)} models finished")

## 4. Results
Per model: median seconds per page, output tokens per second, peak RAM, valid answers, and field F1 on these pages
(only a sanity check: 10 pages are far too few for a score; the reported scores come from 129 dev pages).
Load time = total time minus generation time. The table is also saved to Drive as `results_<device>.csv`.

In [ ]:
import pandas as pd, statistics, matplotlib.pyplot as plt
rows = []
for name, r in results.items():
    d = f"{WORK}/results/{r['folder']}_{DEVICE}"
    pages = [json.loads(l) for l in open(f"{d}/pages.jsonl")]
    s = json.load(open(f"{d}/summary.json"))
    gen = sum(p["seconds"] for p in pages)
    rows.append({"model": name, "device": DEVICE, "pages": len(pages),
                 "s/page (median)": statistics.median(p["seconds"] for p in pages),
                 "tokens/s": round(sum(p["tokens"] for p in pages) / max(gen, 1), 1),
                 "peak RAM GB": round(r["peak_ram_gb"], 1), "load s": round(r["wall_s"] - gen),
                 "valid JSON": f"{s['valid']}/{len(pages)}", "hit token limit": s["hit_token_limit"],
                 "field F1 (10 pages)": s["field_f1"]})
df = pd.DataFrame(rows)
df.to_csv(f"{DRIVE}/results_{DEVICE}.csv", index=False)
display(df)
df.plot.barh(x="model", y="s/page (median)", legend=False, title=f"Seconds per page on {DEVICE}"); plt.tight_layout(); plt.show()
print(f"saved {DRIVE}/results_{DEVICE}.csv")

## 5. Release the runtime
Frees the Colab machine (on a GPU runtime this stops credit use). Run it once the table above is saved.

In [ ]:
from google.colab import runtime
runtime.unassign()